In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import os

In [ ]:
zs = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
snaps = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]

basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (dl)

In [ ]:
norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []

with h5py.File('nexus_outputs.h5', 'r') as f:
    for red in zs:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(f"There are {n} frames")

In [ ]:
fig = plt.figure(figsize=(6, 12)) 

maxmax = np.max(np.log10(norm_dens))
minmin = np.min(np.log10(norm_dens))

for i in range(18):
    ax = fig.add_subplot(6, 3, i+1)
    ax.imshow(np.log10(norm_dens[17-i][:,:,37]), origin="lower", cmap="inferno", vmax = maxmax, vmin = minmin)
    
    ax.set_xticks([])
    ax.set_yticks([])
  
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)  
        spine.set_color('white')  
        
fig.subplots_adjust(wspace=0, hspace=0);

plt.savefig("git_thing/Img/density_evol.png", transparent = True)
os.system("cd /Users/users/roana/roana/BSc_Thesis/git_thing/Img && git add . && git commit -m 'Auto-update from python script' && git push origin master")

In [ ]:
def get_pot(z):

    index = zs.index(z)
    snap_ref = snaps[index]

    density = dens[index]

    density_k = fftn(density)
    power_spec = np.abs(density_k)

    k_1d = 2 * np.pi * fftfreq(res, d=dl)

    kx, ky, kz = np.meshgrid(k_1d, k_1d, k_1d, indexing='ij')
    k_squared = kx**2 + ky**2 + kz**2
    k_squared[0, 0, 0] = 1.0
    phi_k = -density_k / k_squared
    phi_k[0, 0, 0] = 0.0 + 0.0j

    pot = np.real(ifftn(phi_k))

    return pot

In [ ]:
grav_pot = get_pot(0.0)

plt.imshow(grav_pot[:,:,37], origin = "lower", extent=[0,35,0,35])

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

maxmax = np.max(grav_pot)
minmin = np.min(grav_pot)

def update(frame):

    ax.clear()

    slice = frame

    ax.imshow(grav_pot[:,:,slice], origin = "lower", extent=[0,35,0,35], vmax = maxmax, vmin = minmin)
    ax.set_xlabel('cMpc/h')
    ax.set_ylabel('cMpc/h')
    ax.set_title('Gravitational potential at z=0.0')
    ax.set_aspect('equal');

anim = animation.FuncAnimation(fig, update, frames=128, interval=50)

anim.save('pot_slices.mp4', writer='ffmpeg', fps=10)

print("2D Animation successfully saved!")

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

slice = 37

grav_pots = []

for z in zs:
    grav_pots.append(get_pot(z))

grav_pots = np.array(grav_pots)

maxmax = np.max(grav_pots)
minmin = np.min(grav_pots)

def update(frame):

    ax.clear()

    index = frame

    ax.imshow(grav_pots[index-1][:,:,slice], origin = "lower", extent=[0,35,0,35], vmax = maxmax, vmin = minmin)
    ax.set_xlabel('cMpc/h')
    ax.set_ylabel('cMpc/h')
    ax.set_title(f'Gravitational potential at z={zs[index]}; slice {slice}')
    ax.set_aspect('equal');

anim = animation.FuncAnimation(fig, update, frames=18, interval=50)

anim.save('pot_evol.mp4', writer='ffmpeg', fps=5)

print("2D Animation successfully saved!")

In [ ]:
I love Zdeni!
I love Zdeni!
I love Zdeni!
I love Zdeni!
I love Zdeni!
I love Zdeni!
I love Zdeni!
I love Zdeni!
I love Zdeni!